# Chapter 8: Grouping Data with GroupBy, Performing Aggregations, Transformations, and Split-Apply-Combine Patterns

**Companion notebook** for *Beginner's Guide to Pandas* by Ravi Shankar

Run each cell in order. Exercises are at the end.

In [39]:
import pandas as pd
import numpy as np

# Grouping Data with GroupBy, Performing Aggregations, Transformations, and Split-Apply-Combine Patterns

## Understanding the GroupBy Concept

The `groupby()` operation is one of pandas' most powerful features for data analysis. At its core, it implements the **split-apply-combine** pattern:

1. **Split** — Divide your data into logical groups based on one or more criteria
2. **Apply** — Execute a function on each group independently
3. **Combine** — Reassemble the results into a new data structure

This pattern allows you to perform complex analyses without explicit loops, making your code both cleaner and more efficient.

### What is a GroupBy Object?

When you call `.groupby()`, pandas does not immediately compute results. Instead, it returns a **GroupBy object** that represents lazy evaluation — the actual computation happens only when you call an aggregation method:

In [40]:
import pandas as pd
import numpy as np

sales = pd.DataFrame({
    'region': ['North', 'South', 'North', 'East', 'South', 'East'],
    'revenue': [100, 200, 150, 300, 250, 180]
})

# This creates a GroupBy object but computes nothing yet
grouped = sales.groupby('region')
print(type(grouped))
# <class 'pandas.core.groupby.generic.DataFrameGroupBy'>

# The calculation happens only when you call an aggregation method
result = grouped['revenue'].sum()
print(result)

<class 'pandas.core.groupby.generic.DataFrameGroupBy'>
region
East     480
North    250
South    450
Name: revenue, dtype: int64


**Output:**
```
region
East    480
North   250
South   450
Name: revenue, dtype: int64
```

**Why lazy evaluation?** It is more efficient — pandas can optimize the calculation before executing it.

### GroupBy Object Internals

The GroupBy object contains useful metadata about your groups:

In [41]:
grouped = sales.groupby('region')

# Number of groups
print(f"Number of groups: {grouped.ngroups}")

# Dictionary mapping group names to row indices
print(f"Groups dictionary:\n{grouped.groups}")

# Group sizes
print(f"Group sizes:\n{grouped.size()}")

Number of groups: 3
Groups dictionary:
{'East': [3, 5], 'North': [0, 2], 'South': [1, 4]}
Group sizes:
region
East     2
North    2
South    2
dtype: int64


### DataFrameGroupBy vs SeriesGroupBy

The type of GroupBy object depends on what you select:

In [42]:
# DataFrameGroupBy — when you do not select a column
df_grouped = sales.groupby('region')
print(type(df_grouped))
# <class 'pandas.core.groupby.generic.DataFrameGroupBy'>

# SeriesGroupBy — when you select a single column
series_grouped = sales.groupby('region')['revenue']
print(type(series_grouped))
# <class 'pandas.core.groupby.generic.SeriesGroupBy'>

<class 'pandas.core.groupby.generic.DataFrameGroupBy'>
<class 'pandas.core.groupby.generic.SeriesGroupBy'>


---

## Basic GroupBy Operations

### Single Column Grouping

The simplest groupby operation groups data by a single column and applies an aggregation function:

In [43]:
import pandas as pd
import numpy as np

sales_data = pd.DataFrame({
    'region': ['North', 'South', 'North', 'East', 'South', 'East', 'North'],
    'product': ['A', 'B', 'A', 'C', 'B', 'C', 'B'],
    'revenue': [100, 150, 200, 120, 180, 90, 110],
    'quantity': [5, 10, 8, 6, 12, 3, 7]
})

# Group by region and sum revenue
regional_revenue = sales_data.groupby('region')['revenue'].sum()
print(regional_revenue)

region
East     210
North    410
South    330
Name: revenue, dtype: int64


**Output:**
```
region
East     210
North    410
South    330
Name: revenue, dtype: int64
```

### Multiple Column Grouping

When grouping by multiple columns, pandas creates a hierarchical index (MultiIndex):

In [44]:
# Group by region and product
grouped_summary = sales_data.groupby(['region', 'product'])['revenue'].sum()
print(grouped_summary)

region  product
East    C          210
North   A          300
        B          110
South   B          330
Name: revenue, dtype: int64


**Output:**
```
region  product
East    C          210
North   A          300
        B          110
South   B          330
Name: revenue, dtype: int64
```

The result has a **MultiIndex** with two levels (`region` and `product`). Use `.reset_index()` to convert it to a flat DataFrame:

In [45]:
flat_result = grouped_summary.reset_index()
print(flat_result)

  region product  revenue
0   East       C      210
1  North       A      300
2  North       B      110
3  South       B      330


**Output:**
```
  region product  revenue
0   East       C      210
1  North       A      300
2  North       B      110
3  South       B      330
```

### Accessing Individual Groups

You can retrieve a specific group with `.get_group()` or iterate over all groups:

In [46]:
grouped = sales_data.groupby('region')

# Access a specific group
north_data = grouped.get_group('North')
print(north_data)

# Iterate through all groups
for region_name, group_data in grouped:
    print(f"\n{region_name}:")
    print(f"  Transactions: {len(group_data)}")
    print(f"  Total revenue: {group_data['revenue'].sum()}")

  region product  revenue  quantity
0  North       A      100         5
2  North       A      200         8
6  North       B      110         7

East:
  Transactions: 2
  Total revenue: 210

North:
  Transactions: 3
  Total revenue: 410

South:
  Transactions: 2
  Total revenue: 330


---

## Common Aggregation Functions

Pandas provides numerous built-in aggregation functions:

In [47]:
# Individual aggregations
print(sales_data.groupby('region')['revenue'].mean())    # Average
print(sales_data.groupby('region')['revenue'].median())  # Median
print(sales_data.groupby('region')['revenue'].std())     # Standard deviation
print(sales_data.groupby('region')['revenue'].min())     # Minimum
print(sales_data.groupby('region')['revenue'].max())     # Maximum
print(sales_data.groupby('region')['revenue'].count())   # Count of non-null values
print(sales_data.groupby('region').size())               # Total count including nulls

region
East     105.000000
North    136.666667
South    165.000000
Name: revenue, dtype: float64
region
East     105.0
North    110.0
South    165.0
Name: revenue, dtype: float64
region
East     21.213203
North    55.075705
South    21.213203
Name: revenue, dtype: float64
region
East      90
North    100
South    150
Name: revenue, dtype: int64
region
East     120
North    200
South    180
Name: revenue, dtype: int64
region
East     2
North    3
South    2
Name: revenue, dtype: int64
region
East     2
North    3
South    2
dtype: int64


**Key difference:** `count()` excludes NaN values; `size()` includes them.

### Applying Multiple Aggregations with `.agg()`

You can apply multiple aggregation functions simultaneously:

In [48]:
# Multiple aggregations on a single column
revenue_stats = sales_data.groupby('region')['revenue'].agg([
    'sum', 'mean', 'min', 'max', 'count'
])
print(revenue_stats)

        sum        mean  min  max  count
region                                  
East    210  105.000000   90  120      2
North   410  136.666667  100  200      3
South   330  165.000000  150  180      2


**Output:**
```
       sum        mean  min  max  count
region
East   210  105.000000   90  120      2
North  410  136.666667  100  200      3
South  330  165.000000  150  180      2
```

### Different Aggregations for Different Columns

In [49]:
# Apply different functions to different columns
multi_agg = sales_data.groupby('region').agg({
    'revenue': ['sum', 'mean'],
    'quantity': ['sum', 'max']
})
print(multi_agg)

       revenue             quantity    
           sum        mean      sum max
region                                 
East       210  105.000000        9   6
North      410  136.666667       20   8
South      330  165.000000       22  12


### Named Aggregations

Named aggregations provide a cleaner syntax and produce descriptive column names:

In [50]:
# Named aggregations (pandas 0.25+)
result = sales_data.groupby('region').agg(
    total_revenue=('revenue', 'sum'),
    avg_revenue=('revenue', 'mean'),
    total_units=('quantity', 'sum'),
    product_count=('product', 'nunique')
)
print(result)

        total_revenue  avg_revenue  total_units  product_count
region                                                        
East              210   105.000000            9              1
North             410   136.666667           20              2
South             330   165.000000           22              1


**Output:**
```
       total_revenue  avg_revenue  total_units  product_count
region
East             210   105.000000            9              1
North            410   136.666667           20              2
South            330   165.000000           22              1
```

**Why use named aggregations?** Column names are clear and descriptive, easier to reference in downstream code, and better for reports and dashboards.

---

## Understanding `.agg()` vs `.transform()` vs `.filter()`

Before diving into each method, it is important to understand when to use each one:

| Method | Output Shape | Use Case |
|--------|-------------|----------|
| **`.agg()`** | Reduced (one row per group) | Summarize groups to single values |
| **`.transform()`** | Same as input (one row per original row) | Add computed values back to original data |
| **`.filter()`** | Subset of input rows | Keep or remove entire groups based on a condition |

In [51]:
sales = pd.DataFrame({
    'region': ['North', 'North', 'North', 'South', 'South', 'South'],
    'revenue': [100, 150, 200, 80, 120, 90],
    'quantity': [5, 7, 10, 4, 6, 5]
})

# .agg() — reduces to one row per group
agg_result = sales.groupby('region')['revenue'].agg('sum')
print(f".agg() shape: {agg_result.shape}")   # (2,)

# .transform() — returns same shape as input
transform_result = sales.groupby('region')['revenue'].transform('sum')
print(f".transform() shape: {transform_result.shape}")  # (6,)

# .filter() — subsets rows based on group condition
filter_result = sales.groupby('region').filter(lambda x: x['revenue'].sum() > 300)
print(f".filter() shape: {filter_result.shape}")  # (3,)

.agg() shape: (2,)
.transform() shape: (6,)
.filter() shape: (3, 3)


---

## Transformations

Transformations apply functions to each group while **preserving the original DataFrame shape**. This differs from aggregations, which reduce dimensions.

### Basic Transformations

In [52]:
# Calculate the mean revenue per region and broadcast back
regional_mean = sales_data.groupby('region')['revenue'].transform('mean')
print(regional_mean)

0    136.666667
1    165.000000
2    136.666667
3    105.000000
4    165.000000
5    105.000000
6    136.666667
Name: revenue, dtype: float64


**Output:**
```
0    136.666667
1    165.000000
2    136.666667
3    105.000000
4    165.000000
5    105.000000
6    136.666667
Name: revenue, dtype: float64
```

This is useful for creating new columns that represent group-level statistics:

In [53]:
# Add a column showing each row's deviation from its region's mean
sales_data['revenue_deviation'] = (
    sales_data['revenue'] -
    sales_data.groupby('region')['revenue'].transform('mean')
)
print(sales_data[['region', 'revenue', 'revenue_deviation']])

  region  revenue  revenue_deviation
0  North      100         -36.666667
1  South      150         -15.000000
2  North      200          63.333333
3   East      120          15.000000
4  South      180          15.000000
5   East       90         -15.000000
6  North      110         -26.666667


### Common Transform Patterns

In [54]:
# 1. Normalize values within each group (z-score)
sales_data['revenue_zscore'] = sales_data.groupby('region')['revenue'].transform(
    lambda x: (x - x.mean()) / x.std()
)

# 2. Rank within groups
sales_data['revenue_rank'] = sales_data.groupby('region')['revenue'].transform(
    lambda x: x.rank(ascending=False)
)

# 3. Percentage of group total
sales_data['pct_of_region'] = (
    sales_data['revenue'] /
    sales_data.groupby('region')['revenue'].transform('sum') * 100
)

# 4. Cumulative sum within groups
sales_data['cumulative_revenue'] = sales_data.groupby('region')['revenue'].transform('cumsum')

# 5. Fill NaN values with group mean
sales_with_nan = sales_data.copy()
sales_with_nan.loc[0, 'revenue'] = np.nan
sales_with_nan['revenue_filled'] = sales_with_nan.groupby('region')['revenue'].transform(
    lambda x: x.fillna(x.mean())
)

### Custom Transformation Functions

In [55]:
# Normalize values within each group (0-1 scale)
def normalize(x):
    return (x - x.min()) / (x.max() - x.min())

sales_data['revenue_normalized'] = sales_data.groupby('region')['revenue'].transform(normalize)
print(sales_data[['region', 'revenue', 'revenue_normalized']])

  region  revenue  revenue_normalized
0  North      100                 0.0
1  South      150                 0.0
2  North      200                 1.0
3   East      120                 1.0
4  South      180                 1.0
5   East       90                 0.0
6  North      110                 0.1


---

## The Split-Apply-Combine Pattern in Detail

### Using `.apply()` for Custom Logic

The `.apply()` method allows you to execute custom functions on each group. Use it when you need to access multiple columns or perform complex calculations:

In [ ]:
# Calculate revenue concentration (top product's share of total)
def concentration_ratio(group):
    """Calculate what percentage of revenue comes from the top product"""
    return group['revenue'].max() / group['revenue'].sum()

concentration = sales_data.groupby('region').apply(concentration_ratio, include_groups=False)
print(concentration)

### Working with Multiple Columns in Apply

In [ ]:
# Create a summary with multiple calculated metrics
def group_summary(group):
    return pd.Series({
        'total_revenue': group['revenue'].sum(),
        'total_units': group['quantity'].sum(),
        'revenue_per_unit': group['revenue'].sum() / group['quantity'].sum(),
        'num_products': group['product'].nunique(),
        'num_records': len(group)
    })

summary = sales_data.groupby('region').apply(group_summary, include_groups=False)
print(summary)

### `.agg()` vs `.apply()` — When to Use Each

| Method | Use Case | Returns |
|--------|----------|---------|
| **`.agg()`** | Built-in functions (`sum`, `mean`, `max`, etc.) or named aggregations | Aggregated result (one value per group) |
| **`.apply()`** | Custom logic, complex transformations, or returning multiple values | Can return Series, DataFrame, or scalar |

In [58]:
# .agg() with built-in functions (fast, clean)
result_agg = sales_data.groupby('region')['revenue'].agg('sum')

# .apply() with custom function (flexible, but slower)
result_apply = sales_data.groupby('region')['revenue'].apply(lambda x: x.sum())

# Both produce the same result, but .agg() is preferred for simple operations

---

## Filtering Groups

You can filter groups based on group-level conditions using `.filter()`. The function receives each group as a DataFrame and should return `True` to keep the group or `False` to remove it:

In [59]:
# Keep only regions with total revenue > 300
high_revenue_regions = sales_data.groupby('region').filter(
    lambda x: x['revenue'].sum() > 300
)
print(high_revenue_regions)

  region product  revenue  quantity  revenue_deviation  revenue_zscore  \
0  North       A      100         5         -36.666667       -0.665750   
1  South       B      150        10         -15.000000       -0.707107   
2  North       A      200         8          63.333333        1.149932   
4  South       B      180        12          15.000000        0.707107   
6  North       B      110         7         -26.666667       -0.484182   

   revenue_rank  pct_of_region  cumulative_revenue  revenue_normalized  
0           3.0      24.390244                 100                 0.0  
1           2.0      45.454545                 150                 0.0  
2           1.0      48.780488                 300                 1.0  
4           1.0      54.545455                 330                 1.0  
6           2.0      26.829268                 410                 0.1  


**Key point:** `.filter()` returns all rows belonging to qualifying groups — it is all-or-nothing. If a group passes the condition, all its rows are kept; if it fails, all its rows are removed.

### Common Filter Patterns

In [60]:
# Keep only groups with 3 or more transactions
sufficient_data = sales_data.groupby('region').filter(lambda x: len(x) >= 3)

# Keep groups meeting multiple conditions
quality_regions = sales_data.groupby('region').filter(
    lambda x: (x['revenue'].mean() > 100) & (x['quantity'].mean() > 5)
)

# Keep groups with high variance
high_variance = sales_data.groupby('region').filter(
    lambda x: x['revenue'].std() > 40
)

### `.filter()` vs Boolean Indexing

| Aspect | `.filter()` | Boolean Indexing |
|--------|-------------|-----------------|
| **Evaluates** | Condition per group | Condition per row |
| **Returns** | All rows from passing groups | Only matching rows |
| **Use case** | Group-level decisions | Row-level decisions |

In [61]:
# .filter() — group-level condition
# "Keep all rows from regions where average revenue > 150"
method1 = sales_data.groupby('region').filter(
    lambda x: x['revenue'].mean() > 150
)

# Boolean indexing — row-level condition
# "Keep only rows where revenue > 150"
method2 = sales_data[sales_data['revenue'] > 150]

---

## Handling Missing Data in Groups

By default, groupby excludes NaN values from the grouping key:

In [62]:
# Data with missing values
incomplete_data = pd.DataFrame({
    'category': ['A', 'B', None, 'A', 'B', 'A'],
    'value': [10, 20, 30, 15, 25, 20]
})

# NaN category is excluded by default
print(incomplete_data.groupby('category')['value'].sum())

# Include NaN as a group
print(incomplete_data.groupby('category', dropna=False)['value'].sum())

category
A    45
B    45
Name: value, dtype: int64
category
A      45
B      45
NaN    30
Name: value, dtype: int64


NaN values in data columns are excluded from most aggregations by default. Use `count()` to see how many non-NaN values are in each group, and `size()` to see total rows including NaN.

---

## Multi-Level Grouping

### Understanding MultiIndex Results

When you group by multiple columns, pandas creates a **MultiIndex** — a hierarchical index with multiple levels:

In [63]:
import pandas as pd
import numpy as np

np.random.seed(42)
sales = pd.DataFrame({
    'region': np.random.choice(['North', 'South', 'East', 'West'], 12),
    'product': np.random.choice(['Product_A', 'Product_B', 'Product_C'], 12),
    'revenue': np.random.randint(100, 500, 12),
    'quantity': np.random.randint(1, 10, 12)
})

# Group by region and product
multi_group = sales.groupby(['region', 'product'])['revenue'].sum()
print(multi_group)

region  product  
East    Product_A    563
        Product_B    731
        Product_C    748
North   Product_A    352
        Product_B    427
South   Product_B    370
West    Product_A    148
        Product_C    121
Name: revenue, dtype: int64


### Reshaping Multi-Level Results

Use `.reset_index()` to convert the hierarchical index into regular columns:

In [64]:
flat_result = multi_group.reset_index()
print(flat_result)

  region    product  revenue
0   East  Product_A      563
1   East  Product_B      731
2   East  Product_C      748
3  North  Product_A      352
4  North  Product_B      427
5  South  Product_B      370
6   West  Product_A      148
7   West  Product_C      121


Use `.unstack()` to pivot one level of the index into columns, creating a pivot-table-like view:

In [65]:
pivoted = multi_group.unstack(fill_value=0)
print(pivoted)

product  Product_A  Product_B  Product_C
region                                  
East           563        731        748
North          352        427          0
South            0        370          0
West           148          0        121


Use `.stack()` to reverse `.unstack()` — converting columns back to rows:

In [66]:
stacked = pivoted.stack()
print(stacked)

region  product  
East    Product_A    563
        Product_B    731
        Product_C    748
North   Product_A    352
        Product_B    427
        Product_C      0
South   Product_A      0
        Product_B    370
        Product_C      0
West    Product_A    148
        Product_B      0
        Product_C    121
dtype: int64


### Top N Items Per Group

In [67]:
# Get top 2 products by revenue in each region
top_2 = (sales
    .sort_values('revenue', ascending=False)
    .groupby('region', group_keys=False)
    .head(2)
    .reset_index(drop=True)
)
print(top_2)

  region    product  revenue  quantity
0   East  Product_B      444         3
1   East  Product_C      413         3
2  South  Product_B      370         9
3  North  Product_A      352         4
4  North  Product_B      269         7
5   West  Product_A      148         5
6   West  Product_C      121         7


---

## Custom Aggregation Functions

Your custom function receives a **Series** (all values for that group) and should return a **scalar** (single value) when used with `.agg()`:

In [68]:
def range_func(series):
    """Calculate the range (max - min) of a series"""
    return series.max() - series.min()

revenue_range = sales.groupby('region')['revenue'].agg(range_func)
print(revenue_range)

region
East     170
North    194
South      0
West      27
Name: revenue, dtype: int64


### Practical Custom Aggregation Functions

In [69]:
# Interquartile Range (IQR)
def iqr(x):
    return x.quantile(0.75) - x.quantile(0.25)

# Coefficient of Variation (CV)
def cv(x):
    return x.std() / x.mean() if x.mean() != 0 else np.nan

# Count outliers (values > 2 standard deviations from mean)
def count_outliers(x):
    return ((x - x.mean()).abs() > 2 * x.std()).sum()

# Apply these functions
print(sales.groupby('region')['revenue'].agg(iqr))
print(sales.groupby('region')['revenue'].agg(cv))
print(sales.groupby('region')['revenue'].agg(count_outliers))

region
East     106.0
North     97.0
South      0.0
West      13.5
Name: revenue, dtype: float64
region
East     0.211620
North    0.374851
South         NaN
West     0.141947
Name: revenue, dtype: float64
region
East     0
North    0
South    0
West     0
Name: revenue, dtype: int64


### Multiple Aggregations at Once

In [70]:
# Apply multiple functions to the same column
multi_agg = sales.groupby('region')['revenue'].agg([
    'sum',
    'mean',
    'std',
    range_func
])
print(multi_agg)

         sum        mean        std  range_func
region                                         
East    2042  340.333333  72.021293         170
North    779  259.666667  97.336187         194
South    370  370.000000        NaN           0
West     269  134.500000  19.091883          27


---

## Important GroupBy Parameters

### `as_index` — Keep Group Keys as Index or Columns

In [71]:
# Default: as_index=True (group keys become index)
result_indexed = sales.groupby('region', as_index=True)['revenue'].sum()

# as_index=False (group keys become columns, returns DataFrame)
result_flat = sales.groupby('region', as_index=False)['revenue'].sum()

Choose `as_index=False` when you want a regular DataFrame instead of a Series with an index.

### `sort` — Sort Group Keys

In [72]:
# sort=True (default): groups sorted alphabetically
result_sorted = sales.groupby('region', sort=True)['revenue'].sum()

# sort=False: groups in order of appearance
result_unsorted = sales.groupby('region', sort=False)['revenue'].sum()

### `dropna` — Handle NaN Group Keys

In [73]:
# dropna=True (default): NaN groups are excluded
result_drop = sales.groupby('region', dropna=True)['revenue'].sum()

# dropna=False: NaN is treated as a group
result_keep = sales.groupby('region', dropna=False)['revenue'].sum()

### `observed` — Categorical Data

In [74]:
# If your groupby column is categorical, use observed=True to exclude unused categories
sales['region'] = pd.Categorical(sales['region'], categories=['North', 'South', 'East', 'West', 'Unused'])

# observed=False includes empty categories (slower)
result_all = sales.groupby('region', observed=False)['revenue'].sum()

# observed=True only includes categories that appear in data (faster)
result_observed = sales.groupby('region', observed=True)['revenue'].sum()

---

## Advanced Techniques

### Finding Max/Min Rows Per Group

In [ ]:
# Get the row with highest revenue per region using idxmax()
max_idx = sales_data.groupby('region')['revenue'].idxmax()
print(sales_data.loc[max_idx])

# Get top 2 transactions per region
top_2 = sales_data.groupby('region', group_keys=False).apply(
    lambda x: x.nlargest(2, 'revenue'), include_groups=False
)
print(top_2)

### Quantile and Percentile Calculations

In [76]:
# 25th, 50th, 75th percentiles
quantiles = sales_data.groupby('region')['revenue'].quantile([0.25, 0.5, 0.75])
print(quantiles)

# Interquartile Range
q1 = sales_data.groupby('region')['revenue'].quantile(0.25)
q3 = sales_data.groupby('region')['revenue'].quantile(0.75)
iqr = q3 - q1
print(iqr)

region      
East    0.25     97.5
        0.50    105.0
        0.75    112.5
North   0.25    105.0
        0.50    110.0
        0.75    155.0
South   0.25    157.5
        0.50    165.0
        0.75    172.5
Name: revenue, dtype: float64
region
East     15.0
North    50.0
South    15.0
Name: revenue, dtype: float64


### Counting Distinct Values

In [77]:
# Count unique products per region
unique_products = sales_data.groupby('region')['product'].nunique()
print(unique_products)

region
East     1
North    2
South    1
Name: product, dtype: int64


### Conditional Aggregation

In [ ]:
# Sum revenue only where quantity > 5
conditional_sum = sales_data.groupby('region').apply(
    lambda x: x[x['quantity'] > 5]['revenue'].sum(), include_groups=False
)
print(conditional_sum)

### Flattening MultiIndex Column Names

In [79]:
result_multi = sales_data.groupby(['region', 'product']).agg({
    'revenue': ['sum', 'mean', 'count']
})

# Flatten the MultiIndex column names
result_multi.columns = ['_'.join(col).strip() for col in result_multi.columns.values]
print(result_multi)

                revenue_sum  revenue_mean  revenue_count
region product                                          
East   C                210         105.0              2
North  A                300         150.0              2
       B                110         110.0              1
South  B                330         165.0              2


---

## Practical Example: Sales Analysis

Here is a complete example combining multiple groupby techniques:

In [80]:
import pandas as pd
import numpy as np

np.random.seed(42)
sales_data = pd.DataFrame({
    'date': pd.date_range('2024-01-01', periods=12),
    'region': ['North', 'South', 'East', 'West'] * 3,
    'product': ['A', 'B', 'C'] * 4,
    'revenue': [1200, 950, 1100, 800, 1500, 1050, 920, 750, 1300, 1100, 1000, 850],
    'quantity': [10, 8, 9, 7, 12, 9, 8, 6, 11, 9, 8, 7]
})

# Multi-level analysis with named aggregations
analysis = sales_data.groupby(['region', 'product']).agg(
    total_revenue=('revenue', 'sum'),
    avg_revenue=('revenue', 'mean'),
    total_quantity=('quantity', 'sum'),
    transactions=('revenue', 'count')
).round(2)

# Add a calculated column showing revenue share
analysis['revenue_share'] = (
    analysis['total_revenue'] /
    analysis['total_revenue'].sum() * 100
).round(2)

print(analysis)

                total_revenue  avg_revenue  total_quantity  transactions  \
region product                                                             
East   A                  920        920.0               8             1   
       B                 1000       1000.0               8             1   
       C                 1100       1100.0               9             1   
North  A                 1200       1200.0              10             1   
       B                 1500       1500.0              12             1   
       C                 1300       1300.0              11             1   
South  A                 1100       1100.0               9             1   
       B                  950        950.0               8             1   
       C                 1050       1050.0               9             1   
West   A                  800        800.0               7             1   
       B                  750        750.0               6             1   
       C    

### Real-World Pattern: Outlier Detection Per Group

In [81]:
def find_outliers(group):
    Q1 = group.quantile(0.25)
    Q3 = group.quantile(0.75)
    IQR = Q3 - Q1
    return (group < (Q1 - 1.5 * IQR)) | (group > (Q3 + 1.5 * IQR))

sales_data['is_outlier'] = sales_data.groupby('region')['revenue'].transform(find_outliers)
outliers = sales_data[sales_data['is_outlier']]
print(f"Found {len(outliers)} outliers")

Found 0 outliers


### Real-World Pattern: Cohort Analysis

In [82]:
# Assign cohort based on first purchase date in each region
sales_data['cohort'] = sales_data.groupby('region')['date'].transform('min')
print(sales_data[['region', 'date', 'revenue', 'cohort']])

   region       date  revenue     cohort
0   North 2024-01-01     1200 2024-01-01
1   South 2024-01-02      950 2024-01-02
2    East 2024-01-03     1100 2024-01-03
3    West 2024-01-04      800 2024-01-04
4   North 2024-01-05     1500 2024-01-01
5   South 2024-01-06     1050 2024-01-02
6    East 2024-01-07      920 2024-01-03
7    West 2024-01-08      750 2024-01-04
8   North 2024-01-09     1300 2024-01-01
9   South 2024-01-10     1100 2024-01-02
10   East 2024-01-11     1000 2024-01-03
11   West 2024-01-12      850 2024-01-04


---

## Common Pitfalls and How to Avoid Them

### Pitfall 1: Forgetting GroupBy is Lazy

In [83]:
# Wrong: This creates a GroupBy object but computes nothing
grouped = sales_data.groupby('region')  # Nothing happens yet

# Correct: Add an aggregation method
result = sales_data.groupby('region')['revenue'].sum()  # Now it computes

### Pitfall 2: Using `.agg()` When You Need `.transform()`

In [84]:
# Wrong: agg() returns one value per group — shape mismatch when assigning to a column
try:
    sales_data['group_sum'] = sales_data.groupby('region')['revenue'].agg('sum')
except ValueError as e:
    print(f"Error: {e}")

# Correct: transform() broadcasts the result back to original shape
sales_data['group_sum'] = sales_data.groupby('region')['revenue'].transform('sum')

### Pitfall 3: Not Handling NaN Values

In [85]:
import numpy as np

data_with_nan = pd.DataFrame({
    'region': ['North', 'North', 'South', 'South'],
    'revenue': [150, None, 80, 110]
})

# count() excludes NaN; size() includes it — always check which you need
result = data_with_nan.groupby('region')['revenue'].agg(['sum', 'count', 'mean'])
print(result)
# North's count is 1 (NaN excluded), not 2 — this can be misleading

          sum  count   mean
region                     
North   150.0      1  150.0
South   190.0      2   95.0


### Pitfall 4: Forgetting to Reset Index

In [86]:
# Group names become the index by default
result = sales_data.groupby('region')['revenue'].sum()
print(result.index)  # Index(['East', 'North', 'South', 'West'], name='region')

# Use reset_index() if you need 'region' as a regular column
result_df = result.reset_index()
print(result_df)

Index(['East', 'North', 'South', 'West'], dtype='object', name='region')
  region  revenue
0   East     3020
1  North     4000
2  South     3100
3   West     2400


### Pitfall 5: Type Coercion in Mixed-Type Columns

In [87]:
mixed_data = pd.DataFrame({
    'region': ['North', 'South', 'North', 'South'],
    'value': [150, '80', 90, '110']  # Mix of int and str
})

try:
    result = mixed_data.groupby('region')['value'].sum()
except TypeError as e:
    print(f"Error: {e}")

# Solution: Convert to consistent type first
mixed_data['value'] = pd.to_numeric(mixed_data['value'])
result = mixed_data.groupby('region')['value'].sum()
print(result)

region
North    240
South    190
Name: value, dtype: int64


---

## Performance Considerations

Choosing the right method matters for large datasets:

In [88]:
import time

large_sales = pd.DataFrame({
    'region': np.random.choice(['North', 'South', 'East', 'West'], 100000),
    'revenue': np.random.randint(50, 200, 100000)
})

# Fastest: built-in aggregation with string name
start = time.time()
result1 = large_sales.groupby('region')['revenue'].sum()
print(f"Built-in .sum(): {time.time() - start:.6f} seconds")

# Slower: custom lambda function
start = time.time()
result2 = large_sales.groupby('region')['revenue'].agg(lambda x: x.sum())
print(f"Lambda in .agg(): {time.time() - start:.6f} seconds")

# Slowest: .apply()
start = time.time()
result3 = large_sales.groupby('region')['revenue'].apply(lambda x: x.sum())
print(f"Lambda in .apply(): {time.time() - start:.6f} seconds")

Built-in .sum(): 0.002179 seconds
Lambda in .agg(): 0.003396 seconds
Lambda in .apply(): 0.002951 seconds


**Performance tips:**
1. Use built-in string names (`'sum'`, `'mean'`) instead of lambda functions when possible
2. Use a single `.agg()` call with multiple functions instead of separate groupby calls
3. Select the column you need before aggregating: `groupby('region')['revenue'].sum()` rather than `groupby('region').sum()`
4. Use `.agg()` and `.transform()` instead of `.apply()` for simple operations
5. Use `observed=True` for categorical groupby columns

---

## Quick Decision Guide

```
Do you need to...?
│
├─ Reduce data to one row per group? → Use .agg()
│  (e.g., "What is the total per region?")
│
├─ Keep original shape, add computed values? → Use .transform()
│  (e.g., "Add each transaction's % of region total")
│
├─ Keep/remove entire groups based on condition? → Use .filter()
│  (e.g., "Keep only high-performing regions")
│
└─ Complex multi-column or multi-step logic? → Use .apply()
   (e.g., "Calculate revenue per unit using two columns")
```

| Task | Method | Example |
|------|--------|---------|
| Summary statistics | `.agg()` | Total revenue per region |
| Add computed column | `.transform()` | Revenue rank within region |
| Keep/remove groups | `.filter()` | Only regions with revenue > 300 |
| Complex custom logic | `.apply()` | Calculate revenue per item |
| Access specific group | `.get_group()` | Get all North region data |

---

## Exercises

**Exercise 1:** Using the sales data from this chapter, create a report that shows:
1. Total revenue per region (aggregation)
2. Each transaction's percentage of its region's total (transformation)
3. Only regions where the average transaction is above $100 (filtering)

**Solution:**

In [89]:
import pandas as pd

sales = pd.DataFrame({
    'region': ['North', 'South', 'East', 'West', 'North', 'South', 'East', 'West'],
    'revenue': [150, 80, 120, 200, 90, 110, 95, 180]
})

# Step 1: Total revenue per region (aggregation)
region_totals = sales.groupby('region')['revenue'].sum()
print("Step 1 - Total revenue per region:")
print(region_totals)

# Step 2: Percentage of region total (transformation)
sales['pct_of_region'] = (
    sales['revenue'] / sales.groupby('region')['revenue'].transform('sum') * 100
)
print("\nStep 2 - Percentage of region total:")
print(sales[['region', 'revenue', 'pct_of_region']])

# Step 3: Filter for regions with avg transaction > $100
high_avg_regions = sales.groupby('region').filter(
    lambda x: x['revenue'].mean() > 100
)
print("\nStep 3 - Regions with average transaction > $100:")
print(high_avg_regions)

Step 1 - Total revenue per region:
region
East     215
North    240
South    190
West     380
Name: revenue, dtype: int64

Step 2 - Percentage of region total:
  region  revenue  pct_of_region
0  North      150      62.500000
1  South       80      42.105263
2   East      120      55.813953
3   West      200      52.631579
4  North       90      37.500000
5  South      110      57.894737
6   East       95      44.186047
7   West      180      47.368421

Step 3 - Regions with average transaction > $100:
  region  revenue  pct_of_region
0  North      150      62.500000
2   East      120      55.813953
3   West      200      52.631579
4  North       90      37.500000
6   East       95      44.186047
7   West      180      47.368421


**Exercise 2:** Using a dataset of your choice, perform a multi-level groupby analysis:
1. Group by two columns and compute named aggregations
2. Add a column ranking each row within its group
3. Detect outliers (values more than 2 standard deviations from the group mean) using `.transform()`

**Exercise 3:** Write a custom aggregation function that computes the coefficient of variation (standard deviation divided by mean) for each group, and apply it alongside the built-in `sum` and `count` functions in a single `.agg()` call.

---

## Key Takeaways

- **GroupBy implements split-apply-combine** — split data into groups, apply a function, combine results
- **GroupBy objects are lazy** — computation happens only when you call `.agg()`, `.transform()`, `.apply()`, or iterate
- **Use `.agg()` to reduce data** — returns one value per group; use named aggregations for readable output
- **Use `.transform()` to broadcast** — returns the same shape as input, ideal for adding computed columns
- **Use `.apply()` for complex logic** — most flexible, but slower than `.agg()` and `.transform()`
- **Use `.filter()` to subset groups** — all-or-nothing: entire groups pass or fail the condition
- **Handle missing values explicitly** — use the `dropna` parameter to control NaN behavior in group keys
- **Multi-level grouping reveals patterns** — group by multiple columns for hierarchical analysis
- **Performance matters at scale** — prefer built-in string names over lambda functions, and `.agg()` over `.apply()` for simple operations
- **Reset index when needed** — convert group names from index to column with `.reset_index()`

---

# Exercises

Test your understanding of this chapter's concepts.

### Exercise 1: Basic GroupBy Aggregations

You have a dataset of employee records with department, salary, and years of experience. Use GroupBy to calculate summary statistics for each department. Practice using common aggregation functions like mean, max, min, and count.

In [90]:
import pandas as pd

df = pd.DataFrame({
    'employee': ['Alice', 'Bob', 'Carol', 'David', 'Eve', 'Frank', 'Grace', 'Hank'],
    'department': ['HR', 'Engineering', 'Engineering', 'HR', 'Marketing', 'Engineering', 'Marketing', 'HR'],
    'salary': [55000, 95000, 88000, 60000, 72000, 102000, 68000, 58000],
    'years_exp': [3, 7, 5, 8, 4, 10, 2, 6]
})

# TODO: Group the DataFrame by 'department'
dept_groups = None

# TODO: Calculate the mean salary per department
mean_salary = None

# TODO: Calculate the max years_exp per department
max_exp = None

# TODO: Use .agg() to calculate both count and mean of salary in one step
summary = None

print("Mean Salary by Department:")
print(mean_salary)
print("\nMax Experience by Department:")
print(max_exp)
print("\nSummary Table:")
print(summary)

Mean Salary by Department:
None

Max Experience by Department:
None

Summary Table:
None


### Exercise 2: Transform vs Agg: Normalizing Within Groups

You have sales data for different product categories across multiple months. Use .transform() to add a new column that shows each sale as a percentage of its category's total sales. This demonstrates how transform preserves the original DataFrame's shape, unlike agg.

In [91]:
import pandas as pd

df = pd.DataFrame({
    'month': ['Jan', 'Feb', 'Mar', 'Jan', 'Feb', 'Mar', 'Jan', 'Feb', 'Mar'],
    'category': ['Electronics', 'Electronics', 'Electronics', 'Clothing', 'Clothing', 'Clothing', 'Food', 'Food', 'Food'],
    'sales': [12000, 15000, 11000, 8000, 9500, 7000, 4000, 4200, 3800]
})

# TODO: Use .transform('sum') to compute the total sales per category
# and assign it to a new column called 'category_total'
df['category_total'] = None

# TODO: Calculate each row's sales as a percentage of its category total
# Store the result in a new column called 'pct_of_category'
df['pct_of_category'] = None

# TODO: Use .transform('mean') to add a column 'category_mean_sales'
# showing the average sales for each row's category
df['category_mean_sales'] = None

print(df.round(2))

  month     category  sales category_total pct_of_category category_mean_sales
0   Jan  Electronics  12000           None            None                None
1   Feb  Electronics  15000           None            None                None
2   Mar  Electronics  11000           None            None                None
3   Jan     Clothing   8000           None            None                None
4   Feb     Clothing   9500           None            None                None
5   Mar     Clothing   7000           None            None                None
6   Jan         Food   4000           None            None                None
7   Feb         Food   4200           None            None                None
8   Mar         Food   3800           None            None                None


### Exercise 3: Filtering Groups and Custom Aggregations

You have a dataset of customer orders. First, use .filter() to keep only customers who have placed more than 2 orders. Then apply a custom aggregation function to compute the range (max - min) of order values per customer.

In [92]:
import pandas as pd

df = pd.DataFrame({
    'customer_id': [1, 1, 1, 2, 2, 3, 3, 3, 3, 4],
    'order_value': [250, 180, 320, 90, 150, 400, 220, 310, 275, 500],
    'product': ['A', 'B', 'A', 'C', 'A', 'B', 'C', 'A', 'B', 'C']
})

# TODO: Use .filter() to keep only customers who have placed MORE than 2 orders
# Hint: use len(x) > 2 inside the filter lambda
active_customers = None

print("Customers with more than 2 orders:")
print(active_customers)

# TODO: Define a custom aggregation function called 'order_range'
# that returns the difference between the max and min order_value
def order_range(x):
    pass  # TODO: implement this

# TODO: Apply the custom function using .agg() on the filtered DataFrame
# Group by 'customer_id' and aggregate 'order_value' with your custom function
customer_range = None

print("\nOrder value range per customer:")
print(customer_range)

Customers with more than 2 orders:
None

Order value range per customer:
None


### Exercise 4: Multi-Level GroupBy Sales Analysis

You have a retail dataset with region, product category, and quarterly sales figures. Perform a multi-level GroupBy analysis to compute total and average sales by region and category. Then use the split-apply-combine pattern to rank products within each region by total sales.

In [93]:
import pandas as pd

df = pd.DataFrame({
    'region': ['North', 'North', 'North', 'North', 'South', 'South', 'South', 'South', 'West', 'West', 'West', 'West'],
    'category': ['Electronics', 'Clothing', 'Electronics', 'Clothing', 'Electronics', 'Clothing', 'Electronics', 'Clothing', 'Electronics', 'Clothing', 'Electronics', 'Clothing'],
    'quarter': ['Q1', 'Q1', 'Q2', 'Q2', 'Q1', 'Q1', 'Q2', 'Q2', 'Q1', 'Q1', 'Q2', 'Q2'],
    'sales': [23000, 15000, 27000, 13000, 18000, 21000, 22000, 19000, 31000, 11000, 29000, 14000]
})

# TODO: Group by both 'region' and 'category', then calculate total sales
# using .agg() to get both 'sum' and 'mean' for the sales column
region_category_summary = None

print("Sales Summary by Region and Category:")
print(region_category_summary)

# TODO: Group by 'region' and 'category' and compute total sales per group
# Then use .transform('sum') to add a 'total_sales' column to the original df
df['total_sales'] = None

# TODO: Use .transform() with a rank function to add a 'rank_in_region' column
# that ranks categories within each region by their total_sales (rank 1 = highest)
# Hint: group by 'region', then transform the 'total_sales' column using
# lambda x: x.rank(ascending=False, method='dense')
df['rank_in_region'] = None

# Drop duplicate rows to show one row per region-category combination
result = df.drop_duplicates(subset=['region', 'category'])[['region', 'category', 'total_sales', 'rank_in_region']]
result = result.sort_values(['region', 'rank_in_region'])

print("\nCategory Rankings Within Each Region:")
print(result.to_string(index=False))

Sales Summary by Region and Category:
None

Category Rankings Within Each Region:
region    category total_sales rank_in_region
 North Electronics        None           None
 North    Clothing        None           None
 South Electronics        None           None
 South    Clothing        None           None
  West Electronics        None           None
  West    Clothing        None           None


---

# Solutions

*Scroll down only after you've attempted the exercises above.*

<br><br><br><br><br><br><br><br><br><br>

### Solution 1: Basic GroupBy Aggregations

In [94]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    'employee': ['Alice', 'Bob', 'Carol', 'David', 'Eve', 'Frank', 'Grace', 'Hank'],
    'department': ['HR', 'Engineering', 'Engineering', 'HR', 'Marketing', 'Engineering', 'Marketing', 'HR'],
    'salary': [55000, 95000, 88000, 60000, 72000, 102000, 68000, 58000],
    'years_exp': [3, 7, 5, 8, 4, 10, 2, 6]
})

# Group the DataFrame by 'department'
dept_groups = df.groupby('department')

# Calculate the mean salary per department
mean_salary = dept_groups['salary'].mean()

# Calculate the max years_exp per department
max_exp = dept_groups['years_exp'].max()

# Use .agg() to calculate both count and mean of salary in one step
summary = dept_groups['salary'].agg(['count', 'mean'])

print("Mean Salary by Department:")
print(mean_salary)
print("\nMax Experience by Department:")
print(max_exp)
print("\nSummary Table:")
print(summary)

Mean Salary by Department:
department
Engineering    95000.000000
HR             57666.666667
Marketing      70000.000000
Name: salary, dtype: float64

Max Experience by Department:
department
Engineering    10
HR              8
Marketing       4
Name: years_exp, dtype: int64

Summary Table:
             count          mean
department                      
Engineering      3  95000.000000
HR               3  57666.666667
Marketing        2  70000.000000


### Solution 2: Transform vs Agg: Normalizing Within Groups

In [95]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    'month': ['Jan', 'Feb', 'Mar', 'Jan', 'Feb', 'Mar', 'Jan', 'Feb', 'Mar'],
    'category': ['Electronics', 'Electronics', 'Electronics', 'Clothing', 'Clothing', 'Clothing', 'Food', 'Food', 'Food'],
    'sales': [12000, 15000, 11000, 8000, 9500, 7000, 4000, 4200, 3800]
})

# Use .transform('sum') to compute the total sales per category
# and assign it to a new column called 'category_total'
df['category_total'] = df.groupby('category')['sales'].transform('sum')

# Calculate each row's sales as a percentage of its category total
# Store the result in a new column called 'pct_of_category'
df['pct_of_category'] = (df['sales'] / df['category_total']) * 100

# Use .transform('mean') to add a column 'category_mean_sales'
# showing the average sales for each row's category
df['category_mean_sales'] = df.groupby('category')['sales'].transform('mean')

print(df.round(2))

  month     category  sales  category_total  pct_of_category  \
0   Jan  Electronics  12000           38000            31.58   
1   Feb  Electronics  15000           38000            39.47   
2   Mar  Electronics  11000           38000            28.95   
3   Jan     Clothing   8000           24500            32.65   
4   Feb     Clothing   9500           24500            38.78   
5   Mar     Clothing   7000           24500            28.57   
6   Jan         Food   4000           12000            33.33   
7   Feb         Food   4200           12000            35.00   
8   Mar         Food   3800           12000            31.67   

   category_mean_sales  
0             12666.67  
1             12666.67  
2             12666.67  
3              8166.67  
4              8166.67  
5              8166.67  
6              4000.00  
7              4000.00  
8              4000.00  


### Solution 3: Filtering Groups and Custom Aggregations

In [96]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    'customer_id': [1, 1, 1, 2, 2, 3, 3, 3, 3, 4],
    'order_value': [250, 180, 320, 90, 150, 400, 220, 310, 275, 500],
    'product': ['A', 'B', 'A', 'C', 'A', 'B', 'C', 'A', 'B', 'C']
})

# Use .filter() to keep only customers who have placed MORE than 2 orders
active_customers = df.groupby('customer_id').filter(lambda x: len(x) > 2)

print("Customers with more than 2 orders:")
print(active_customers)

# Define a custom aggregation function called 'order_range'
# that returns the difference between the max and min order_value
def order_range(x):
    return x.max() - x.min()

# Apply the custom function using .agg() on the filtered DataFrame
# Group by 'customer_id' and aggregate 'order_value' with your custom function
customer_range = active_customers.groupby('customer_id')['order_value'].agg(order_range)

print("\nOrder value range per customer:")
print(customer_range)

Customers with more than 2 orders:
   customer_id  order_value product
0            1          250       A
1            1          180       B
2            1          320       A
5            3          400       B
6            3          220       C
7            3          310       A
8            3          275       B

Order value range per customer:
customer_id
1    140
3    180
Name: order_value, dtype: int64


### Solution 4: Multi-Level GroupBy Sales Analysis

In [97]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    'region': ['North', 'North', 'North', 'North', 'South', 'South', 'South', 'South', 'West', 'West', 'West', 'West'],
    'category': ['Electronics', 'Clothing', 'Electronics', 'Clothing', 'Electronics', 'Clothing', 'Electronics', 'Clothing', 'Electronics', 'Clothing', 'Electronics', 'Clothing'],
    'quarter': ['Q1', 'Q1', 'Q2', 'Q2', 'Q1', 'Q1', 'Q2', 'Q2', 'Q1', 'Q1', 'Q2', 'Q2'],
    'sales': [23000, 15000, 27000, 13000, 18000, 21000, 22000, 19000, 31000, 11000, 29000, 14000]
})

# Group by both 'region' and 'category', then calculate total sales
# using .agg() to get both 'sum' and 'mean' for the sales column
region_category_summary = df.groupby(['region', 'category'])['sales'].agg(['sum', 'mean'])

print("Sales Summary by Region and Category:")
print(region_category_summary)

# Group by 'region' and 'category' and compute total sales per group
# Then use .transform('sum') to add a 'total_sales' column to the original df
df['total_sales'] = df.groupby(['region', 'category'])['sales'].transform('sum')

# Use .transform() with a rank function to add a 'rank_in_region' column
# that ranks categories within each region by their total_sales (rank 1 = highest)
df['rank_in_region'] = df.groupby('region')['total_sales'].transform(
    lambda x: x.rank(ascending=False, method='dense')
)

# Drop duplicate rows to show one row per region-category combination
result = df.drop_duplicates(subset=['region', 'category'])[['region', 'category', 'total_sales', 'rank_in_region']]
result = result.sort_values(['region', 'rank_in_region'])

print("\nCategory Rankings Within Each Region:")
print(result.to_string(index=False))

Sales Summary by Region and Category:
                      sum     mean
region category                   
North  Clothing     28000  14000.0
       Electronics  50000  25000.0
South  Clothing     40000  20000.0
       Electronics  40000  20000.0
West   Clothing     25000  12500.0
       Electronics  60000  30000.0

Category Rankings Within Each Region:
region    category  total_sales  rank_in_region
 North Electronics        50000             1.0
 North    Clothing        28000             2.0
 South Electronics        40000             1.0
 South    Clothing        40000             1.0
  West Electronics        60000             1.0
  West    Clothing        25000             2.0
